In [1]:
# Importa a biblioteca específica do Colab para acessar o Google Drive
import pandas as pd
import numpy as np
import glob
import os
import time
import tensorflow as tf

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from tensorflow import keras

from sklearn.metrics import classification_report, confusion_matrix

from keras.models import Sequential
from keras.layers import Dense, Dropout
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
# Definindo os caminho da nova pasta no PC pessoal
caminho_pasta_tratado = '../dataset tratado/cicids2017/Sem Redução de Dimensionalidade/'

nome_dados_treinamento = 'cicids2017_treinamento.csv'
nome_dados_teste = 'cicids2017_teste.csv'

In [3]:
# 1. Carregando o input a partir dos CSVs de dados JÁ TRATADOS
print("Carregando os datasets tratados em CSV...")
df_treino = pd.read_csv(caminho_pasta_tratado + "/" + nome_dados_treinamento)
df_teste = pd.read_csv(caminho_pasta_tratado + "/" + nome_dados_teste)

print(f"Dados carregados! Treino: {df_treino.shape} | Teste: {df_teste.shape}")

display(df_treino.head())
display(df_teste.head())

Carregando os datasets tratados em CSV...
Dados carregados! Treino: (1979513, 79) | Teste: (848363, 79)


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,0.855604,3.229166e-05,0.000009,0.000003,2.945736e-06,9.153974e-09,0.001531,0.000000,0.002132,0.003079,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
1,0.006760,2.272266e-03,0.000009,0.000017,0.000000e+00,4.576987e-09,0.000000,0.000000,0.000000,0.000000,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
2,0.848402,1.333333e-07,0.000005,0.000000,9.302326e-07,0.000000e+00,0.000242,0.002581,0.001010,0.000000,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
3,0.000809,1.233333e-06,0.000005,0.000007,6.821705e-06,1.830795e-07,0.001773,0.018925,0.007406,0.000000,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN
4,0.000809,1.449567e-03,0.000005,0.000007,4.496124e-06,3.509023e-07,0.001168,0.012473,0.004881,0.000000,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,BENIGN


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,0.585107,9.205005e-02,0.000000,0.000017,4.651163e-07,4.576987e-08,0.000242,0.002581,0.001010,0.000000,...,1.0,0.000342,0.000000,0.000342,0.000342,0.091667,0.000000,0.091667,0.091667,BENIGN
1,0.001221,1.276342e-03,0.000009,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000,...,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,BENIGN
2,0.006760,5.102475e-01,0.000086,0.000062,1.924031e-04,1.157062e-05,0.024013,0.000000,0.020889,0.026176,...,1.0,0.001497,0.003769,0.006687,0.000459,0.083290,0.000132,0.083333,0.083136,BENIGN
3,0.000809,9.841666e-06,0.000005,0.000007,7.286822e-06,3.356457e-07,0.001894,0.020215,0.007911,0.000000,...,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,BENIGN
4,0.650904,4.666666e-07,0.000000,0.000003,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000,...,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,BENIGN


In [4]:
# 0. Separando as características (X) do gabarito/rótulo (y)
X_treino = df_treino.drop('Label', axis=1)
y_treino = df_treino['Label']

X_teste = df_teste.drop('Label', axis=1)
y_teste = df_teste['Label']

In [6]:
# 1. Convertendo as Labels de Texto para Números (Requisito do TensorFlow)
print("Codificando as labels para a Rede Neural...")
encoder = LabelEncoder()
y_treino_num = encoder.fit_transform(y_treino)
y_teste_num = encoder.transform(y_teste)
num_classes = len(encoder.classes_) # Descobre quantos tipos de ataques existem

Codificando as labels para a Rede Neural...


In [9]:

# 2. Construindo a Arquitetura da Rede Neural Profunda (DNN)
# Uma arquitetura clássica com duas camadas ocultas
dnn_model = Sequential([
    # Camada Oculta 1 (128 neurônios) conectada à entrada (78 features)
    keras.layers.Input(shape=(X_treino.shape[1],)),
    Dense(128, activation='relu'),
    Dropout(0.2), # Dropout desliga neurônios aleatórios para evitar "vício" (overfitting)

    # Camada Oculta 2 (64 neurônios)
    Dense(64, activation='relu'),
    Dropout(0.2),

    # Camada de Saída (Um neurônio para cada classe de ataque, usando Softmax para dar a probabilidade)
    Dense(num_classes, activation='softmax')
])

In [10]:

# 3. Compilando o modelo (Definindo como ele vai calcular os erros e se ajustar)
dnn_model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

# 4. O Treinamento (Medindo o Custo Computacional)
print("\nIniciando o treinamento da Rede Neural Profunda (DNN)...")
inicio_treino_dnn = time.time()

# O modelo treinará por 10 rodadas completas (epochs), processando os dados em lotes (batch_size)
history = dnn_model.fit(X_treino, y_treino_num, epochs=10, batch_size=256, validation_split=0.1)

fim_treino_dnn = time.time()
tempo_treinamento_dnn = fim_treino_dnn - inicio_treino_dnn
print(f"\nTreinamento DNN concluído! Tempo total: {tempo_treinamento_dnn:.4f} segundos.")


Iniciando o treinamento da Rede Neural Profunda (DNN)...
Epoch 1/10
6960/6960 ━━━━━━━━━━━━━━━━━━━━ 11s 2ms/step - accuracy: 0.9532 - loss: 0.1363 - val_accuracy: 0.9734 - val_loss: 0.0666
Epoch 2/10
6960/6960 ━━━━━━━━━━━━━━━━━━━━ 9s 1ms/step - accuracy: 0.9748 - loss: 0.0641 - val_accuracy: 0.9789 - val_loss: 0.0478
Epoch 3/10
6960/6960 ━━━━━━━━━━━━━━━━━━━━ 9s 1ms/step - accuracy: 0.9780 - loss: 0.0521 - val_accuracy: 0.9797 - val_loss: 0.0434
Epoch 4/10
6960/6960 ━━━━━━━━━━━━━━━━━━━━ 9s 1ms/step - accuracy: 0.9792 - loss: 0.0480 - val_accuracy: 0.9806 - val_loss: 0.0432
Epoch 5/10
6960/6960 ━━━━━━━━━━━━━━━━━━━━ 9s 1ms/step - accuracy: 0.9799 - loss: 0.0461 - val_accuracy: 0.9821 - val_loss: 0.0407
Epoch 6/10
6960/6960 ━━━━━━━━━━━━━━━━━━━━ 9s 1ms/step - accuracy: 0.9805 - loss: 0.0448 - val_accuracy: 0.9812 - val_loss: 0.0399
Epoch 7/10
6960/6960 ━━━━━━━━━━━━━━━━━━━━ 9s 1ms/step - accuracy: 0.9809 - loss: 0.0439 - val_accuracy: 0.9822 - val_loss: 0.0401
Epoch 8/10
6960/6960 ━━━━━━━━━━

In [14]:
# 5. Classificação dos Dados de Teste
print("\nIniciando a classificação dos dados de teste...")
inicio_teste_dnn = time.time()

# A rede retorna a probabilidade para cada classe. O argmax pega a classe com a maior probabilidade.
previsoes_probabilidades = dnn_model.predict(X_teste)
previsoes_dnn_num = np.argmax(previsoes_probabilidades, axis=1)

fim_teste_dnn = time.time()
tempo_teste_dnn = fim_teste_dnn - inicio_teste_dnn
print(f"Classificação concluída! Tempo de previsão: {tempo_teste_dnn:.4f} segundos.")

# Revertendo os números das previsões de volta para os nomes originais (Benign, DoS, etc.)
previsoes_dnn_labels = encoder.inverse_transform(previsoes_dnn_num)

# 6. Avaliação dos Resultados
print("\n=== MATRIZ DE CONFUSÃO (DNN) ===")
labels = sorted(np.unique(np.concatenate([y_teste.values, previsoes_dnn_labels])))
cm = confusion_matrix(y_teste, previsoes_dnn_labels, labels=labels)

cm_df = pd.DataFrame(
    cm,
    index=[f"Real_{lbl}" for lbl in labels],
    columns=[f"Pred_{lbl}" for lbl in labels]
)

def color_confusion_matrix(data):
    styles = pd.DataFrame("", index=data.index, columns=data.columns)

    # Green diagonal (correct predictions)
    for i in range(min(data.shape[0], data.shape[1])):
        styles.iat[i, i] = "background-color: #c6efce; color: #006100; font-weight: bold;"

    # Red off-diagonal only when value != 0 (errors)
    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            if i != j and data.iat[i, j] != 0:
                styles.iat[i, j] = "background-color: #ffc7ce; color: #9c0006; font-weight: bold;"

    return styles

display(
    cm_df.style
        .apply(color_confusion_matrix, axis=None)
        .format("{:.0f}")
)

print("\n=== RELATÓRIO DE MÉTRICAS (Precisão, Recall, F1-Score) ===")
print(classification_report(y_teste, previsoes_dnn_labels, zero_division=0))


Iniciando a classificação dos dados de teste...
26512/26512 ━━━━━━━━━━━━━━━━━━━━ 12s 466us/step
Classificação concluída! Tempo de previsão: 17.3258 segundos.

=== MATRIZ DE CONFUSÃO (DNN) ===


,Pred_BENIGN,Pred_Bot,Pred_DDoS,Pred_DoS GoldenEye,Pred_DoS Hulk,Pred_DoS Slowhttptest,Pred_DoS slowloris,Pred_FTP-Patator,Pred_Heartbleed,Pred_Infiltration,Pred_PortScan,Pred_SSH-Patator,Pred_Web Attack � Brute Force,Pred_Web Attack � Sql Injection,Pred_Web Attack � XSS
Real_BENIGN,671956,13,15,25,3737,30,12,1,0,0,5197,27,0,0,1
Real_Bot,354,210,0,0,0,0,0,0,0,0,0,0,0,0,0
Real_DDoS,527,0,37776,3,1,0,0,0,0,0,0,0,0,0,0
Real_DoS GoldenEye,85,0,0,3074,0,2,2,0,0,0,0,0,0,0,0
Real_DoS Hulk,76,0,0,1,69190,0,0,0,0,0,0,0,0,0,0
Real_DoS Slowhttptest,17,0,0,0,0,1626,14,0,0,0,0,1,0,0,0
Real_DoS slowloris,11,0,0,1,1,37,1704,2,0,0,0,1,0,0,0
Real_FTP-Patator,0,0,0,0,2,0,4,2320,0,0,0,0,0,0,0
Real_Heartbleed,3,0,0,0,0,0,0,0,3,0,0,0,0,0,0
Real_Infiltration,10,0,0,0,0,0,0,0,0,0,0,0,0,0,0



=== RELATÓRIO DE MÉTRICAS (Precisão, Recall, F1-Score) ===
                            precision    recall  f1-score   support

                    BENIGN       0.99      0.99      0.99    681014
                       Bot       0.94      0.37      0.53       564
                      DDoS       1.00      0.99      0.99     38307
             DoS GoldenEye       0.99      0.97      0.98      3163
                  DoS Hulk       0.95      1.00      0.97     69267
          DoS Slowhttptest       0.96      0.98      0.97      1658
             DoS slowloris       0.98      0.97      0.98      1757
               FTP-Patator       1.00      1.00      1.00      2326
                Heartbleed       1.00      0.50      0.67         6
              Infiltration       0.00      0.00      0.00        10
                  PortScan       0.90      0.94      0.92     47836
               SSH-Patator       0.96      0.89      0.92      1821
  Web Attack � Brute Force       0.93      0.03      0.

In [ ]:
# 7. Exportacao das tabelas em formato LaTeX
from IPython.display import Markdown, display

CLASS_ALIASES_LATEX = {'BENIGN': 'BENIGN', 'Bot': 'Bot', 'DDoS': 'DDoS', 'DoS GoldenEye': 'DoS GE', 'DoS Hulk': 'Hulk', 'DoS Slowhttptest': 'Slowhttp', 'DoS slowloris': 'Slowloris', 'FTP-Patator': 'FTP', 'Heartbleed': 'Heart', 'Infiltration': 'Infil', 'PortScan': 'PortScan', 'SSH-Patator': 'SSH', 'Web Attack - Brute Force': 'Brute', 'Web Attack - Sql Injection': 'SQL', 'Web Attack - XSS': 'XSS', 'Desconhecido': 'Desconh'}


def escape_latex(value):
    replacements = {
        "\\": r"\textbackslash{}",
        "&": r"\&",
        "%": r"\%",
        "$": r"\$",
        "#": r"\#",
        "_": r"\_",
        "{": r"\{",
        "}": r"\}",
        "~": r"\textasciitilde{}",
        "^": r"\textasciicircum{}",
    }
    return "".join(replacements.get(char, char) for char in str(value))


def latex_class_name(label, short=False):
    if short:
        return escape_latex(CLASS_ALIASES_LATEX.get(label, label))
    return escape_latex(label)


def format_confusion_value(value, is_diagonal):
    value = int(value)
    if is_diagonal:
        return rf"\ok{{{value}}}"
    if value != 0:
        return rf"\err{{{value}}}"
    return "0"


def make_latex_confusion_matrix(cm_values, class_labels, caption, table_label):
    headers = [latex_class_name(label, short=True) for label in class_labels]
    rows = []
    for i, real_label in enumerate(class_labels):
        row_values = [format_confusion_value(cm_values[i][j], i == j) for j in range(len(class_labels))]
        rows.append((rf"Real\_{latex_class_name(real_label, short=True)}", row_values))

    first_col_width = max([0] + [len(row_name) for row_name, _ in rows])
    col_widths = [max(len(headers[i]), *(len(values[i]) for _, values in rows)) for i in range(len(headers))]

    def format_row(first_cell, values):
        first = first_cell.ljust(first_col_width)
        rest = " & ".join(str(value).ljust(col_widths[i]) for i, value in enumerate(values))
        return f"            {first} & {rest} \\\\"

    lines = [
        r"\begin{table}[H]",
        r"    \centering",
        r"    \scriptsize",
        r"    \resizebox{\linewidth}{!}{",
        rf"        \begin{{tabular}}{{l|{'r' * len(class_labels)}}}",
        r"            \hline",
        format_row("", headers),
        r"            \hline",
    ]
    lines.extend(format_row(row_name, row_values) for row_name, row_values in rows)
    lines.extend([
        r"            \hline",
        r"        \end{tabular}",
        r"    }",
        rf"    \caption{{{escape_latex(caption)}}}",
        rf"    \label{{{table_label}}}",
        r"\end{table}",
    ])
    return "\n".join(lines)


def format_metric(value):
    return "-" if value is None else f"{value:.2f}"


def make_latex_metrics_report(y_true_values, y_pred_values, class_labels, caption, table_label):
    report = classification_report(
        y_true_values,
        y_pred_values,
        labels=class_labels,
        output_dict=True,
        zero_division=0,
    )
    total_support = int(sum(report[label]["support"] for label in class_labels))
    rows = []
    for label in class_labels:
        metrics = report[label]
        rows.append([
            latex_class_name(label),
            format_metric(metrics["precision"]),
            format_metric(metrics["recall"]),
            format_metric(metrics["f1-score"]),
            str(int(metrics["support"])),
        ])

    rows.extend([
        [r"\textbf{Acur?cia}", "-", format_metric(report["accuracy"]), "-", str(total_support)],
        [r"\textbf{M?dia Macro}", format_metric(report["macro avg"]["precision"]), format_metric(report["macro avg"]["recall"]), format_metric(report["macro avg"]["f1-score"]), str(total_support)],
        [r"\textbf{M?dia Ponderada}", format_metric(report["weighted avg"]["precision"]), format_metric(report["weighted avg"]["recall"]), format_metric(report["weighted avg"]["f1-score"]), str(total_support)],
    ])

    headers = ["Classe", "Precis?o", "Revoca??o", "F1-score", "Suporte"]
    col_widths = [max(len(str(row[i])) for row in [headers] + rows) for i in range(len(headers))]

    def format_row(values):
        return "        " + " & ".join(str(value).ljust(col_widths[i]) for i, value in enumerate(values)) + r" \\"

    lines = [
        r"\begin{table}[H]",
        r"    \centering",
        r"    \small",
        r"    \begin{tabular}{lrrrr}",
        r"        \hline",
        format_row(headers),
        r"        \hline",
    ]
    lines.extend(format_row(row) for row in rows[:len(class_labels)])
    lines.extend([
        r"        \hline",
        format_row(rows[-3]),
        format_row(rows[-2]),
        format_row(rows[-1]),
        r"        \hline",
        r"    \end{tabular}",
        rf"    \caption{{{escape_latex(caption)}}}",
        rf"    \label{{{table_label}}}",
        r"\end{table}",
    ])
    return "\n".join(lines)


labels_latex = labels
y_true_latex = y_teste
y_pred_latex = previsoes_dnn_labels

latex_matriz_confusao = make_latex_confusion_matrix(
    cm,
    labels_latex,
    "Matriz de Confus?o - DNN",
    "table:dnn_completo_mc",
)
latex_relatorio_metricas = make_latex_metrics_report(
    y_true_latex,
    y_pred_latex,
    labels_latex,
    "Relat?rio de M?tricas do Modelo DNN",
    "table:dnn_completo_metricas",
)

print(latex_matriz_confusao)
print("\n")
print(latex_relatorio_metricas)
